# Tree-Based Hessian Estimation | RA Research Notebook (2026-W43)

**Research direction (October 15 meeting with Nathan Wycoff):** Extend the first-order tree-based gradient estimator in *Regression Trees Know Calculus* (Section 3.1, Equation 3) to second-order derivatives, using two successive tree levels and four grandchildren. Start with the basic estimator and experiments, before physics-informed optimization or power-grid applications.

**Sources:** [Wycoff, Regression Trees Know Calculus, arXiv:2405.13846v4](https://arxiv.org/abs/2405.13846); Nathan Wycoff meeting 2026-10-15 (approximately 13:52–14:06); Nocedal and Wright, *Numerical Optimization*.

**Status:** Research derivation and checks, **not** a proved general Hessian estimator. Ordinary CART predictions are piecewise constant, so this estimates the derivatives of a hypothesized underlying smooth function, not the usual pointwise Hessian of the fitted staircase.

## 1. Assumptions and notation

Let \(f:\mathbb R^p\to\mathbb R\), with observations \(y_n=f(x_n)+\epsilon_n\). Assume enough smoothness and, when indicated, exact uniform-design regional means. At internal node \(i\), let \(\sigma_i\) be its splitting coordinate, \(l^i,u^i\) its bounds, and \(\hat\mu_l^i,\hat\mu_r^i\) its child response means.

The paper's **first-order estimator** is

\[
\gamma_i=\frac{2(\hat\mu_r^i-\hat\mu_l^i)}{u_{\sigma_i}^i-l_{\sigma_i}^i}
\approx\frac{\partial f}{\partial x_{\sigma_i}}. \tag{Paper Eq. 3}
\]

Its interpretation assumes the cell is sufficiently small and the local derivative varies little. For second derivatives, shrinking widths alone is insufficient because estimated means also have noise.

In this notebook: (i) **balanced** means midpoint splits; (ii) **exact cell means** mean normalized integrals of \(f\); (iii) unequal splits and finite samples are investigated separately.

## 2. Diagonal Hessian entry from repeated splits along \(x_1\)

Let a parent span \([l,u]\) with width \(W=u-l\). Split it at its midpoint into \(L\) and \(R\). Split **both children again along \(x_1\)**, each at its midpoint. The four grandchildren (left to right) are \(LL,LR,RL,RR\).

For the left and right child, apply the original equation separately:

\[
\gamma_L=\frac{2(\hat\mu_{LR}-\hat\mu_{LL})}{W/2}
=\frac{4(\hat\mu_{LR}-\hat\mu_{LL})}{W},
\quad
\gamma_R=\frac{4(\hat\mu_{RR}-\hat\mu_{RL})}{W}.
\]

The child centers are separated by \(W/2\), hence

\[
\widetilde H_{11}
=\frac{\gamma_R-\gamma_L}{W/2}
=\boxed{\frac{8}{W^2}
(\hat\mu_{RR}-\hat\mu_{RL}-\hat\mu_{LR}+\hat\mu_{LL})}.
\tag{A}
\]

This is a **candidate second-derivative estimator for balanced geometry**. The outer denominator is the separation between the locations of the two slope estimates, *not* the width \(W/4\) of a grandchild.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from numpy.polynomial.legendre import leggauss
BLUE, GOLD, RED = "#2874a6", "#bd7f33", "#bb4c44"

def diagram_diagonal():
    fig,(tree,intervals)=plt.subplots(2,1,figsize=(9,5),
                                      gridspec_kw={"height_ratios":[1.3,1]})
    pos={"P":(2,2),"L":(1,1),"R":(3,1),
         "LL":(.5,0),"LR":(1.5,0),"RL":(2.5,0),"RR":(3.5,0)}
    for parent,kid in [("P","L"),("P","R"),("L","LL"),("L","LR"),
                       ("R","RL"),("R","RR")]:
        tree.plot([pos[parent][0],pos[kid][0]],
                  [pos[parent][1]-.12,pos[kid][1]+.12],color=".5",lw=1.5)
    for name,(x,y) in pos.items():
        color=RED if name=="P" else (BLUE if name.startswith("L") else GOLD)
        tree.add_patch(Rectangle((x-.3,y-.15),.6,.3,facecolor=color,
                                 edgecolor=color,alpha=.25))
        tree.text(x,y,name,ha="center",va="center",fontsize=12)
    tree.set(xlim=(0,4),ylim=(-.25,2.3),title="Tree topology: parent, children, grandchildren")
    tree.axis("off")
    for j,name in enumerate(["LL","LR","RL","RR"]):
        intervals.add_patch(Rectangle((j/4,.2),.25,.56,facecolor=BLUE if j<2 else GOLD,
                                      edgecolor="white",alpha=.28))
        intervals.text((j+.5)/4,.47,name,ha="center",va="center",fontsize=12)
    intervals.annotate("",xy=(.75,.9),xytext=(.25,.9),
                       arrowprops={"arrowstyle":"<->"})
    intervals.text(.5,.94,"distance between child centers = W/2",ha="center",fontsize=10)
    intervals.set(xlim=(0,1),ylim=(0,1.12),xticks=[0,.25,.5,.75,1],
                  xlabel="x1 (parent is [0,1])",yticks=[])
    intervals.spines[["top","left","right"]].set_visible(False)
    fig.tight_layout();plt.show()

diagram_diagonal()

## 3. Verify with exact regional means

For a uniform-design interval \(I=[a,b]\),
\[
\mu_I=\frac{1}{b-a}\int_a^b f(x)\,dx.
\]
Compute this integral using Gaussian quadrature rather than substituting the cell midpoint (which is not equivalent for nonlinear \(f\)).

For \(f(x)=x^2\) on \([0,1]\), the four means must be
\[
(\mu_{LL},\mu_{LR},\mu_{RL},\mu_{RR})=(1,7,19,37)/48.
\]
Equation (A) then gives \(8(37-19-7+1)/48=2=f''(x)\).

In [ ]:
def mean_1d(f,a,b,q=32):
    z,w=leggauss(q)
    x=(a+b)/2+(b-a)*z/2
    return float(np.dot(w,f(x))/2)

def four_balanced_means(f,center=.5,width=1.):
    e=np.linspace(center-width/2,center+width/2,5)
    return np.array([mean_1d(f,e[i],e[i+1]) for i in range(4)])

def balanced_h11(means,width=1.):
    LL,LR,RL,RR=means
    return 8*(RR-RL-LR+LL)/width**2

quadratic_means=four_balanced_means(lambda x:x**2)
print("Exact x^2 means:",quadratic_means)
print("H11 estimate:",balanced_h11(quadratic_means),"true:",2)
assert np.allclose(quadratic_means,np.array([1,7,19,37])/48)
assert np.isclose(balanced_h11(quadratic_means),2,atol=1e-12)

### Bias check with a fourth-degree function

For a sufficiently smooth \(f\), the *balanced uniform-cell-mean* stencil has the expansion, at parent midpoint \(c\),

\[
\widetilde H_{11}(c)
=f''(c)+\frac{W^2}{64}f^{(4)}(c)+O(W^4). \tag{B}
\]

This follows from Taylor expansion and requires more smoothness than the paper's original first-derivative consistency assumptions. For \(f(x)=x^4\), \(c=.5\), \(W=1\): true \(f''(c)=3\), predicted bias \(24/64=3/8\), hence \(\widetilde H_{11}=27/8=3.375\).

In [ ]:
fourth_means=four_balanced_means(lambda x:x**4)
fourth_est=balanced_h11(fourth_means)
print("x^4 estimate:",fourth_est,"true:",3,"bias:",fourth_est-3)
assert np.isclose(fourth_est,27/8,atol=1e-12)
assert np.isclose(fourth_est-3,24/64,atol=1e-12)

widths=np.array([.9,.7,.5,.35,.25,.18,.12])
errors=np.array([balanced_h11(four_balanced_means(np.exp,.5,w),w)-np.exp(.5)
                 for w in widths])
fig,ax=plt.subplots(figsize=(7,3.7))
ax.loglog(widths,np.abs(errors),"o-",label="Observed |error| for exp(x)")
ax.loglog(widths,np.exp(.5)*widths**2/64,"--",
          label="Predicted leading bias magnitude")
ax.set(xlabel="Parent width W",ylabel="Absolute error",
       title="Balanced geometry: second-derivative bias is order W²")
ax.legend();ax.grid(True,which="both",alpha=.25);plt.show()

## 4. Critical counterexample: unequal child split thresholds

Ordinary CART chooses RSS-optimal thresholds, not necessarily geometric midpoints. Suppose the parent is \([l,u]\), splits at \(t\), its left child splits at \(s_L\), and its right child splits at \(s_R\):

\[
l<s_L<t<s_R<u,\quad W_L=t-l,\quad W_R=u-t,\quad W=u-l.
\]

A direct recursive use of the paper's formula gives

\[
\gamma_L=\frac{2(\hat\mu_{LR}-\hat\mu_{LL})}{W_L},\quad
\gamma_R=\frac{2(\hat\mu_{RR}-\hat\mu_{RL})}{W_R}.
\]

Dividing their difference by the child-center separation \(W/2\) yields

\[
\widetilde H_{11}^{naive}
=\frac4W\left[
\frac{\hat\mu_{RR}-\hat\mu_{RL}}{W_R}
-\frac{\hat\mu_{LR}-\hat\mu_{LL}}{W_L}
\right]. \tag{C}
\]

**But the two first-derivative estimates need not correspond to the child centers when the grandchildren are unequally sized.** This can produce nonvanishing bias, even for \(f=x^2\).

In [ ]:
def unequal_h11(f,edges):
    l,sL,t,sR,u=map(float,edges)
    assert l<sL<t<sR<u
    means=np.array([mean_1d(f,a,b) for a,b in zip(edges[:-1],edges[1:])])
    LL,LR,RL,RR=means
    gamma_L=2*(LR-LL)/(t-l)
    gamma_R=2*(RR-RL)/(u-t)
    return (gamma_R-gamma_L)/((u-l)/2),means,(gamma_L,gamma_R)

edges=[0,1/8,1/2,7/8,1]
unequal,means,gradients=unequal_h11(lambda x:x**2,edges)
print("Edges:",edges,"means:",means,"gradients:",gradients)
print("Naive H11:",unequal,"true H11:",2,"absolute error:",abs(unequal-2))
assert np.isclose(unequal,7/3,atol=1e-12)

fig,ax=plt.subplots(figsize=(8,2.2))
for j,(a,b) in enumerate(zip(edges[:-1],edges[1:])):
    ax.add_patch(Rectangle((a,.2),b-a,.6,facecolor=BLUE if j<2 else GOLD,
                           edgecolor="black",alpha=.22))
    ax.text((a+b)/2,.5,["LL","LR","RL","RR"][j],ha="center",va="center")
ax.set(xlim=(-.02,1.02),ylim=(0,1),yticks=[],xticks=edges,
       title="Unequal thresholds change effective derivative locations")
plt.show()

A possible **alternative** (not the original TBGE) is a local quadratic reconstruction from four cell means: estimate \(\beta\) in
\[
\mu_{[a,b]}\approx\beta_0+\beta_1 E_{[a,b]}[X-c]
+\beta_2 E_{[a,b]}[(X-c)^2],
\]
and report \(2\hat\beta_2\). This accounts for cell widths via known uniform moments. Test it on exactly the same uneven example. No claim is made here about general CART consistency.

In [ ]:
def quadratic_moment_h11(f,edges):
    center=(edges[0]+edges[-1])/2
    A=[];y=[]
    for a,b in zip(edges[:-1],edges[1:]):
        ex=(a+b)/2
        ex2=(a*a+a*b+b*b)/3
        A.append([1,ex-center,ex2-2*center*ex+center**2])
        y.append(mean_1d(f,a,b))
    beta=np.linalg.lstsq(np.asarray(A),np.asarray(y),rcond=None)[0]
    return 2*beta[2]

corrected=quadratic_moment_h11(lambda x:x**2,edges)
print("Moment-based quadratic H11:",corrected)
assert np.isclose(corrected,2,atol=1e-12)

## 5. Mixed partial \(H_{12}=\partial_{x_1}\partial_{x_2}f\)

**Different geometry:** split the parent along \(x_1\), then split **both** children along \(x_2\) at the same relative position (balanced and aligned). Denote the resulting quadrants \(LD,LU,RD,RU\), where L/R indicate \(x_1\) and D/U indicate \(x_2\).

If the parent widths are \(W_1,W_2\), the paper's estimates of \(\partial f/\partial x_2\) in the left and right children are

\[
\gamma_{L,2}=\frac{2(\mu_{LU}-\mu_{LD})}{W_2},\qquad
\gamma_{R,2}=\frac{2(\mu_{RU}-\mu_{RD})}{W_2}.
\]

The two \(x_2\)-slopes are separated by \(W_1/2\) along \(x_1\), giving

\[
\widetilde H_{12}
=\frac{\gamma_{R,2}-\gamma_{L,2}}{W_1/2}
=\boxed{\frac{4(\mu_{RU}-\mu_{RD}-\mu_{LU}+\mu_{LD})}{W_1W_2}}.
\tag{D}
\]

This estimates one mixed partial under the specific geometry, not an entire Hessian from a generic two-level subtree.

In [ ]:
def diagram_mixed():
    fig,ax=plt.subplots(figsize=(6,5))
    for ix in (0,1):
        for iy in (0,1):
            lab=("L" if ix==0 else "R")+("D" if iy==0 else "U")
            ax.add_patch(Rectangle((ix/2,iy/2),.5,.5,
                 facecolor=BLUE if ix==0 else GOLD,edgecolor="white",alpha=.23))
            ax.text(ix/2+.25,iy/2+.25,lab,ha="center",va="center",fontsize=15)
    ax.axvline(.5,color=RED,lw=3,label="First split: x1")
    ax.axhline(.5,color=BLUE,lw=2,ls="--",label="Second-level splits: x2")
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="x1",ylabel="x2",
           title="Four grandchildren for a mixed partial")
    ax.set_aspect("equal");ax.legend(loc="upper center",bbox_to_anchor=(.5,-.14))
    fig.tight_layout();plt.show()

diagram_mixed()

In [ ]:
def mean_2d(f,x0,x1,y0,y1,q=16):
    z,w=leggauss(q)
    X=(x0+x1)/2+(x1-x0)*z/2
    Y=(y0+y1)/2+(y1-y0)*z/2
    return float(np.sum(f(X[:,None],Y[None,:])*w[:,None]*w[None,:])/4)

def mixed_quadrant_means(f):
    return { "LD":mean_2d(f,0,.5,0,.5),
             "LU":mean_2d(f,0,.5,.5,1),
             "RD":mean_2d(f,.5,1,0,.5),
             "RU":mean_2d(f,.5,1,.5,1)}

def balanced_h12(means,W1=1,W2=1):
    return 4*(means["RU"]-means["RD"]-means["LU"]+means["LD"])/(W1*W2)

f=lambda x1,x2:x1**2+3*x1*x2+2*x2**2
quad=mixed_quadrant_means(f)
print("Quadrant means:",quad)
print("Estimated H12:",balanced_h12(quad),"true H12:",3)
assert np.allclose([quad[k] for k in ["LD","LU","RD","RU"]],
                   [7/16,29/16,21/16,55/16],atol=1e-12)
assert np.isclose(balanced_h12(quad),3,atol=1e-12)

For the test function \(f(x_1,x_2)=x_1^2+3x_1x_2+2x_2^2\), its analytic Hessian is
\[
\nabla^2f=\begin{pmatrix}2&3\\3&4\end{pmatrix}.
\]
The four quadrant means are \((7,29,21,55)/16\) in order \(LD,LU,RD,RU\), producing \(\widetilde H_{12}=3\). Diagonal entries require **separate** repeated-coordinate stencils. Analytically \(H_{12}=H_{21}\); separately estimated noisy entries need not coincide.

## 6. Statistical error amplification

In a fixed balanced one-dimensional partition, assume the four estimated means have **independent additive errors** with variance \(\sigma^2/n\), where \(n\) is observations per grandchild. Equation (A) then implies

\[
\operatorname{Var}(\widetilde H_{11})
=4\left(\frac8{W^2}\right)^2\frac{\sigma^2}{n}
=\frac{256\sigma^2}{nW^4},\quad
\operatorname{SD}(\widetilde H_{11})=\frac{16\sigma}{W^2\sqrt n}. \tag{E}
\]

In this simplified fixed-partition model, \(nW^4\to\infty\) suffices for the variance to vanish. If uniform 1D sampling gives \(n\asymp NW\), that becomes \(NW^5\to\infty\). This calculation **does not** establish a convergence rate for data-adaptively selected CART partitions.

In [ ]:
rng=np.random.default_rng(20261021)
sigma,n,W,reps=.35,400,.6,20000
e=rng.normal(0,sigma/np.sqrt(n),size=(reps,4))
noise=8*(e[:,3]-e[:,2]-e[:,1]+e[:,0])/W**2
predicted_sd=16*sigma/(W**2*np.sqrt(n))
print("Observed SD:",noise.std(ddof=1),"predicted SD:",predicted_sd)
assert abs(noise.std(ddof=1)/predicted_sd-1)<.04
fig,ax=plt.subplots(figsize=(7,3))
ax.hist(noise,bins=55,density=True,alpha=.7)
ax.axvline(predicted_sd,color=RED,ls="--",label="+1 theoretical SD")
ax.axvline(-predicted_sd,color=RED,ls="--",label="-1 theoretical SD")
ax.set(xlabel="Hessian estimation noise",ylabel="Density",
       title="Noise amplification from differencing four sample means")
ax.legend();plt.show()

## 7. Research conclusions and next implementation milestone

**Established in the source:** Wycoff's Eq. (3) and Algorithm 1 provide a first-derivative estimate using one split direction at each internal node.

**Requested at the October 15 meeting:** Extend first derivatives to second order by combining two local derivative estimates using four grandchildren, and build a first working algorithm. The simplest starting point may impose two successive splits on the same feature.

**Derived here, not proven for general CART:** balanced diagonal stencil (A), balanced mixed stencil (D), smooth balanced bias expansion (B), and fixed-partition noise formula (E). Counterexample (C) shows that a naive nonmidpoint extension can fail even on a quadratic.

**Unresolved research questions:** What is the correct spatial normalization for uneven splits? How should derivative locations account for design density? Which splitting patterns suffice to identify every Hessian component? How do sample size and data-adaptive split selection change the error? What should the algorithm do when a necessary second-level split does not exist?

**Next observable deliverable:** Implement one function that accepts a parent node, its children and grandchildren, their split-feature IDs, bounds, and means. Return a second-order estimate only when the required geometry is present. Unit-test on known quadratic Hessians; report unavailable components rather than setting them silently to zero. Keep this PICART/RA work separate from the F1 independent study.

**Research workflow:** derivation → test on known derivatives → counterexamples → generalized geometry → validation under genuine CART → later power-system application.